<a href="https://colab.research.google.com/github/jamalinu/amazigh-nlp-spacy/blob/main/advanced_morphology_tamazight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install spacy --quiet

import spacy
from spacy.lang.xx import MultiLanguage   # base multilenguaje
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex
from spacy import displacy
import re

print('✅ spaCy versión:', spacy.__version__)
print('✅ Entorno listo para Tarifit NLP')

✅ spaCy versión: 3.8.11
✅ Entorno listo para Tarifit NLP


## 📚 PASO 2 — Morfología Bereber: Teoría de Clíticos

### ¿Qué es un clítico?
Un clítico es un morfema que fonológicamente depende de
una palabra adyacente (su "huésped"), pero gramaticalmente
funciona como palabra independiente.

### Tipos de clíticos en Tarifit

| Tipo       | Posición  | Función              | Ejemplo              |
|------------|-----------|----------------------|----------------------|
| Proclítico | ANTES     | Preposición/negación | d=, ur=, γur=        |
| Enclítico  | DESPUÉS   | Pronombre objeto     | =iyi, =ak, =as       |
| Mesoclítico| EN MEDIO  | Objeto + dirección   | =d=, =nn=            |

### Estructura de la palabra verbal en Tarifit

  [NEG] + [PRON.SUJETO] + RAÍZ VERBAL + [ASPECTO] + [PRON.OBJETO]
  
  Ejemplo:  ur + i + ffiγ + =ara  =  "yo no salí"
             NEG  S1SG  salir  NEG2

In [2]:
# ============================================================
# PASO 2 — Diccionario de clíticos Tarifit
# Organizamos los clíticos en categorías lingüísticas.
# Esto será la base de nuestro tokenizer personalizado.
# ============================================================

# PROCLITICOS: van PEGADOS al inicio de la palabra
# El símbolo "=" indica el punto de unión (boundary)
PROCLITICS = {
    # Preposiciones
    "d":   "PREP",    # con / y
    "γur": "PREP",    # en casa de / donde
    "xf":  "PREP",    # sobre / acerca de
    "ger": "PREP",    # entre
    "s":   "PREP",    # hacia / con (instrumental)
    "n":   "GEN",     # de (genitivo)
    "i":   "DAT",     # a / para (dativo)
    # Negación
    "ur":  "NEG",     # no (negación verbal)
    "ulac":"NEG",     # no hay / sin
    # Artículo/Estado
    "w":   "STATE",   # estado libre masculino
    "t":   "STATE",   # estado libre femenino
}

# ENCLITICOS: van PEGADOS al final de la palabra
ENCLITICS = {
    # Pronombres objeto directo
    "iyi":  "PRO.1SG",   # me (1ª sing.)
    "ak":   "PRO.2MSG",  # te (2ª masc. sing.)
    "am":   "PRO.2FSG",  # te (2ª fem. sing.)
    "as":   "PRO.3SG",   # le/lo/la (3ª sing.)
    "aγ":   "PRO.1PL",   # nos (1ª plur.)
    "awen": "PRO.2MPL",  # os (2ª masc. plur.)
    "asent":"PRO.3FPL",  # les (3ª fem. plur.)
    # Partículas
    "ara":  "NEG2",      # negación 2ª parte (ur...ara)
    "nni":  "DEM",       # ese/aquel (demostrativo distal)
    "a":    "DEM",       # este (demostrativo proximal)
    "d":    "DIR",       # venitive (movimiento hacia aquí)
    "nn":   "ANAPH",     # anafórico
}

# SEPARADOR estándar en lingüística bereber
CLITIC_BOUNDARY = "="

print("✅ Procliticos cargados:", len(PROCLITICS))
print("✅ Encliticos cargados:", len(ENCLITICS))
print()
print("Ejemplo de palabra con clíticos:")
print("  ur=ffiγ=ara")
print("  ur  = NEG (proclítico)")
print("  ffiγ = salir (raíz verbal)")
print("  ara = NEG2 (enclítico)")
print("  → 'no salí'")

✅ Procliticos cargados: 11
✅ Encliticos cargados: 12

Ejemplo de palabra con clíticos:
  ur=ffiγ=ara
  ur  = NEG (proclítico)
  ffiγ = salir (raíz verbal)
  ara = NEG2 (enclítico)
  → 'no salí'


## ✂️ PASO 3 — Tokenizer Personalizado para Clíticos

### ¿Por qué necesitamos un tokenizer especial?

El tokenizer por defecto de spaCy separa por espacios y
puntuación. En Tarifit, los clíticos se escriben UNIDOS
a la palabra con "=" o directamente sin separador.

### Problema:
  Input:  "urdffiγara"       ← todo junto, sin espacios
  spaCy normal: ["urdffiγara"]  ← 1 token (¡incorrecto!)
  
### Solución:
  Nuestro tokenizer: ["ur", "ffiγ", "ara"]  ← 3 tokens ✅
  Con etiquetas:      [NEG,  VERB,   NEG2]

In [4]:
# ============================================================
# PASO 3 — Tokenizer personalizado para Tarifit
# Estrategia:
#   1. Detectar procliticos al INICIO de la palabra
#   2. Detectar encliticos al FINAL de la palabra
#   3. Extraer la raíz que queda en el CENTRO
#   4. Devolver lista de tokens con sus etiquetas
# ============================================================

class TarifitTokenizer:
    """
    Tokenizer lingüístico para Tarifit (Bereber del Rif).
    Separa clíticos pegados a la raíz verbal o nominal.
    """

    def __init__(self, proclitics: dict, enclitics: dict):
        self.proclitics = proclitics   # dict {forma: etiqueta}
        self.enclitics  = enclitics    # dict {forma: etiqueta}

        # Ordenar por longitud descendente: primero los más largos
        # Esto evita que "ur" consuma parte de "ulac"
        self.sorted_pro = sorted(proclitics.keys(),
                                  key=len, reverse=True)
        self.sorted_enc = sorted(enclitics.keys(),
                                  key=len, reverse=True)

    def tokenize(self, word: str):
        """
        Recibe una palabra (posiblemente con clíticos unidos)
        y devuelve lista de tuplas (forma, etiqueta).
        """
        tokens = []
        remaining = word.lower().strip()

        # --- 1. Separar por "=" si existe boundary explícito ---
        if "=" in remaining:
            parts = remaining.split("=")
            for part in parts:
                if not part:
                    continue
                label = self._identify(part)
                tokens.append((part, label))
            return tokens

        # --- 2. Detectar PROCLITICOS al inicio ---
        found_pro = True
        while found_pro:
            found_pro = False
            for pro in self.sorted_pro:
                if remaining.startswith(pro) and len(remaining) > len(pro):
                    tokens.append((pro, self.proclitics[pro]))
                    remaining = remaining[len(pro):]
                    found_pro = True
                    break  # reiniciar búsqueda desde el inicio

        # --- 3. Detectar ENCLITICOS al final ---
        enc_tokens = []
        found_enc = True
        while found_enc:
            found_enc = False
            for enc in self.sorted_enc:
                if remaining.endswith(enc) and len(remaining) > len(enc):
                    enc_tokens.insert(0, (enc, self.enclitics[enc]))
                    remaining = remaining[:-len(enc)]
                    found_enc = True
                    break

        # --- 4. Lo que queda es la RAÍZ ---
        if remaining:
            tokens.append((remaining, "ROOT"))

        tokens.extend(enc_tokens)
        return tokens

    def _identify(self, part: str):
        """Identifica la etiqueta de un segmento."""
        if part in self.proclitics:
            return self.proclitics[part]
        if part in self.enclitics:
            return self.enclitics[part]
        return "ROOT"

    def tokenize_sentence(self, sentence: str):
        """Tokeniza una oración completa palabra por palabra."""
        words = sentence.split()
        all_tokens = []
        for word in words:
            result = self.tokenize(word)
            all_tokens.extend(result)
        return all_tokens


# --- Instanciar el tokenizer ---
tarifit_tok = TarifitTokenizer(PROCLITICS, ENCLITICS)

# --- Test básico ---
test_words = [
    "ur=ffiγ=ara",       # no salí (con boundaries explícitos)
    "urffiγara",         # no salí (sin boundaries)
    "γurnekkni",         # en casa de nosotros
    "sskrsas",           # lo hizo bajar (causativo + obj)
    "daswali",           # y lo vio (venitive + obj + verbo)
]

print("=" * 50)
print("🧪 TEST DEL TOKENIZER TARIFIT")
print("=" * 50)
for word in test_words:
    result = tarifit_tok.tokenize(word)
    print(f"\n📝 Input:  '{word}'")
    for forma, etiqueta in result:
        print(f"   ├─ '{forma}'  →  {etiqueta}")

🧪 TEST DEL TOKENIZER TARIFIT

📝 Input:  'ur=ffiγ=ara'
   ├─ 'ur'  →  NEG
   ├─ 'ffiγ'  →  ROOT
   ├─ 'ara'  →  NEG2

📝 Input:  'urffiγara'
   ├─ 'ur'  →  NEG
   ├─ 'ffiγ'  →  ROOT
   ├─ 'ara'  →  NEG2

📝 Input:  'γurnekkni'
   ├─ 'γur'  →  PREP
   ├─ 'n'  →  GEN
   ├─ 'ekkni'  →  ROOT

📝 Input:  'sskrsas'
   ├─ 's'  →  PREP
   ├─ 's'  →  PREP
   ├─ 'krs'  →  ROOT
   ├─ 'as'  →  PRO.3SG

📝 Input:  'daswali'
   ├─ 'd'  →  PREP
   ├─ 'aswali'  →  ROOT


In [5]:
# ============================================================
# Test con oraciones completas en Tarifit
# ============================================================

oraciones = [
    "ur ffiγ ara ddarwi",
    "γurnekkni d axxam amuqqran",
    "sskrs as tafunast",
]

glosas = [
    "NEG salí NEG2 juntos → 'No salimos juntos'",
    "en.casa.de=nosotros y casa grande → 'En nuestra casa grande'",
    "hizo.bajar 3SG.OBJ vaca → 'Le bajó la vaca'",
]

print("=" * 55)
print("📖 ANÁLISIS DE ORACIONES TARIFIT")
print("=" * 55)

for oracion, glosa in zip(oraciones, glosas):
    print(f"\n🗣️  '{oracion}'")
    print(f"📌  Glosa: {glosa}")
    tokens = tarifit_tok.tokenize_sentence(oracion)
    print(f"✂️  Tokens:")
    for forma, etiqueta in tokens:
        print(f"     {forma:<12} [{etiqueta}]")

📖 ANÁLISIS DE ORACIONES TARIFIT

🗣️  'ur ffiγ ara ddarwi'
📌  Glosa: NEG salí NEG2 juntos → 'No salimos juntos'
✂️  Tokens:
     ur           [ROOT]
     ffiγ         [ROOT]
     ar           [ROOT]
     a            [DEM]
     d            [PREP]
     d            [PREP]
     arwi         [ROOT]

🗣️  'γurnekkni d axxam amuqqran'
📌  Glosa: en.casa.de=nosotros y casa grande → 'En nuestra casa grande'
✂️  Tokens:
     γur          [PREP]
     n            [GEN]
     ekkni        [ROOT]
     d            [ROOT]
     axx          [ROOT]
     am           [PRO.2FSG]
     amuqqran     [ROOT]

🗣️  'sskrs as tafunast'
📌  Glosa: hizo.bajar 3SG.OBJ vaca → 'Le bajó la vaca'
✂️  Tokens:
     s            [PREP]
     s            [PREP]
     krs          [ROOT]
     as           [ROOT]
     t            [STATE]
     afunast      [ROOT]


## 🏷️ PASO 4 — Analizador Morfológico (POS Tagging)

### ¿Qué es POS Tagging?
Asignar a cada token su categoría gramatical:
VERB, NOUN, PREP, PRON, ADJ, NEG...

### Sistema de etiquetas para Tarifit

| Etiqueta | Descripción          | Ejemplo Tarifit  |
|----------|----------------------|------------------|
| VERB     | Verbo (raíz verbal)  | ffiγ, ili, kker  |
| NOUN     | Nombre libre         | axxam, tafunast  |
| PREP     | Preposición clítica  | d=, s=, γur=     |
| PRON     | Pronombre enclítico  | =iyi, =as, =ak   |
| NEG      | Negación             | ur=, =ara        |
| ADJ      | Adjetivo             | amuqqran, amẓyan |
| DET      | Determinante         | =nni, =a         |
| GEN      | Genitivo             | n=               |

### Morfología del nombre bereber
Los nombres tienen ESTADO LIBRE y ESTADO ANEXO:
  axxam  (libre)  → yixf n uxxam  (anexo: uxxam)
  afus   (libre)  → tizi n wafus  (anexo: wafus)

In [6]:
# ============================================================
# PASO 4 — Analizador Morfológico para Tarifit
#
# Estrategia basada en REGLAS LINGÜÍSTICAS:
#   - Patrones de prefijos nominales (a-, ta-, i-, ti-)
#   - Patrones verbales por aspecto (aoristo, perfecto)
#   - Tabla de formas conocidas (lexicón pequeño)
# ============================================================

# --- Lexicón base de Tarifit ---
LEXICON = {
    # Verbos (forma de aoristo/base)
    "ffiγ":    {"pos": "VERB", "gloss": "salir",      "aspect": "PERF"},
    "kker":    {"pos": "VERB", "gloss": "levantarse",  "aspect": "AOR"},
    "kkres":   {"pos": "VERB", "gloss": "desatar",     "aspect": "AOR"},
    "sskrs":   {"pos": "VERB", "gloss": "hacer bajar", "aspect": "CAUS"},
    "wali":    {"pos": "VERB", "gloss": "ver",         "aspect": "AOR"},
    "ili":     {"pos": "VERB", "gloss": "ser/estar",   "aspect": "AOR"},
    "čča":     {"pos": "VERB", "gloss": "comer",       "aspect": "AOR"},
    "sw":      {"pos": "VERB", "gloss": "beber",       "aspect": "AOR"},
    "ruḥ":     {"pos": "VERB", "gloss": "ir",          "aspect": "AOR"},
    "uγal":    {"pos": "VERB", "gloss": "volver",      "aspect": "AOR"},

    # Nombres masculinos (prefijo a-)
    "axxam":   {"pos": "NOUN", "gloss": "casa",        "gender": "M", "state": "libre"},
    "afus":    {"pos": "NOUN", "gloss": "mano",        "gender": "M", "state": "libre"},
    "aγyul":   {"pos": "NOUN", "gloss": "burro",       "gender": "M", "state": "libre"},
    "arba":    {"pos": "NOUN", "gloss": "niño",        "gender": "M", "state": "libre"},
    "aẓar":    {"pos": "NOUN", "gloss": "pie/raíz",    "gender": "M", "state": "libre"},
    "amuqqran":{"pos": "ADJ",  "gloss": "grande",      "gender": "M"},
    "amẓyan":  {"pos": "ADJ",  "gloss": "pequeño",     "gender": "M"},

    # Nombres femeninos (prefijo ta-...-t)
    "tafunast":{"pos": "NOUN", "gloss": "vaca",        "gender": "F", "state": "libre"},
    "taddart": {"pos": "NOUN", "gloss": "casa/pueblo", "gender": "F", "state": "libre"},
    "tamγart": {"pos": "NOUN", "gloss": "mujer",       "gender": "F", "state": "libre"},
    "tafat":   {"pos": "NOUN", "gloss": "luz",         "gender": "F", "state": "libre"},

    # Pronombres independientes
    "nekkni":  {"pos": "PRON", "gloss": "nosotros",    "person": "1PL"},
    "nkk":     {"pos": "PRON", "gloss": "yo",          "person": "1SG"},
    "keyyi":   {"pos": "PRON", "gloss": "tú (masc.)",  "person": "2MSG"},
    "netta":   {"pos": "PRON", "gloss": "él",          "person": "3MSG"},
    "nettat":  {"pos": "PRON", "gloss": "ella",        "person": "3FSG"},

    # Partículas
    "ddarwi":  {"pos": "ADV",  "gloss": "juntos"},
    "εawd":    {"pos": "ADV",  "gloss": "de nuevo"},
    "sin":     {"pos": "NUM",  "gloss": "dos"},
    "kraḍ":    {"pos": "NUM",  "gloss": "tres"},
}

# --- Patrones morfológicos por prefijos nominales ---
NOMINAL_PATTERNS = [
    (r"^a[^aeiou]",   "NOUN", "M.libre"),   # a + C... → masculino libre
    (r"^ta.+t$",      "NOUN", "F.libre"),   # ta...t   → femenino libre
    (r"^i[^aeiou]",   "NOUN", "M.plural"),  # i + C... → plural masculino
    (r"^ti.+in$",     "NOUN", "F.plural"),  # ti...in  → plural femenino
    (r"^u[^aeiou]",   "NOUN", "M.anexo"),   # u + C... → estado anexo masc.
    (r"^w[aeiou]",    "NOUN", "M.anexo"),   # w + V... → estado anexo masc.
]

# --- Patrones verbales por aspecto ---
VERBAL_PATTERNS = [
    (r"^tt.+",  "VERB", "IMPERF"),    # tt- → imperfectivo
    (r"^ss.+",  "VERB", "CAUS"),      # ss- → causativo
    (r"^mm.+",  "VERB", "RECIP"),     # mm- → recíproco
    (r"^m.+",   "VERB", "PASS"),      # m-  → pasivo/medio
    (r"^tt.+",  "VERB", "REFL"),      # tt- → reflexivo (contexto)
]


class TarifitMorphAnalyzer:
    """
    Analizador morfológico para Tarifit.
    Combina lexicón, patrones nominales y verbales.
    """

    def __init__(self, lexicon, nominal_patterns, verbal_patterns):
        self.lexicon   = lexicon
        self.nom_pats  = nominal_patterns
        self.verb_pats = verbal_patterns

    def analyze(self, token: str, clitic_label: str = None):
        """
        Analiza un token y devuelve su análisis morfológico completo.
        Si viene con etiqueta de clítico, la respeta.
        """
        result = {
            "token":  token,
            "pos":    "UNK",
            "gloss":  "?",
            "morph":  {},
            "source": "unknown"
        }

        # 1. Si tiene etiqueta de clítico, usarla directamente
        if clitic_label and clitic_label not in ("ROOT", "UNK"):
            result["pos"]    = self._clitic_to_pos(clitic_label)
            result["morph"]  = {"clitic_type": clitic_label}
            result["source"] = "clitic_dict"
            result["gloss"]  = self._clitic_gloss(token)
            return result

        # 2. Buscar

In [8]:
# ============================================================
# PASO 4 — Analizador Morfológico para Tarifit
#
# Estrategia basada en REGLAS LINGÜÍSTICAS:
#   - Patrones de prefijos nominales (a-, ta-, i-, ti-)
#   - Patrones verbales por aspecto (aoristo, perfecto)
#   - Tabla de formas conocidas (lexicón pequeño)
# ============================================================

# --- Lexicón base de Tarifit ---
LEXICON = {
    # Verbos (forma de aoristo/base)
    "ffiγ":    {"pos": "VERB", "gloss": "salir",      "aspect": "PERF"},
    "kker":    {"pos": "VERB", "gloss": "levantarse",  "aspect": "AOR"},
    "kkres":   {"pos": "VERB", "gloss": "desatar",     "aspect": "AOR"},
    "sskrs":   {"pos": "VERB", "gloss": "hacer bajar", "aspect": "CAUS"},
    "wali":    {"pos": "VERB", "gloss": "ver",         "aspect": "AOR"},
    "ili":     {"pos": "VERB", "gloss": "ser/estar",   "aspect": "AOR"},
    "čča":     {"pos": "VERB", "gloss": "comer",       "aspect": "AOR"},
    "sw":      {"pos": "VERB", "gloss": "beber",       "aspect": "AOR"},
    "ruḥ":     {"pos": "VERB", "gloss": "ir",          "aspect": "AOR"},
    "uγal":    {"pos": "VERB", "gloss": "volver",      "aspect": "AOR"},

    # Nombres masculinos (prefijo a-)
    "axxam":   {"pos": "NOUN", "gloss": "casa",        "gender": "M", "state": "libre"},
    "afus":    {"pos": "NOUN", "gloss": "mano",        "gender": "M", "state": "libre"},
    "aγyul":   {"pos": "NOUN", "gloss": "burro",       "gender": "M", "state": "libre"},
    "arba":    {"pos": "NOUN", "gloss": "niño",        "gender": "M", "state": "libre"},
    "aẓar":    {"pos": "NOUN", "gloss": "pie/raíz",    "gender": "M", "state": "libre"},
    "amuqqran":{"pos": "ADJ",  "gloss": "grande",      "gender": "M"},
    "amẓyan":  {"pos": "ADJ",  "gloss": "pequeño",     "gender": "M"},

    # Nombres femeninos (prefijo ta-...-t)
    "tafunast":{"pos": "NOUN", "gloss": "vaca",        "gender": "F", "state": "libre"},
    "taddart": {"pos": "NOUN", "gloss": "casa/pueblo", "gender": "F", "state": "libre"},
    "tamγart": {"pos": "NOUN", "gloss": "mujer",       "gender": "F", "state": "libre"},
    "tafat":   {"pos": "NOUN", "gloss": "luz",         "gender": "F", "state": "libre"},

    # Pronombres independientes
    "nekkni":  {"pos": "PRON", "gloss": "nosotros",    "person": "1PL"},
    "nkk":     {"pos": "PRON", "gloss": "yo",          "person": "1SG"},
    "keyyi":   {"pos": "PRON", "gloss": "tú (masc.)",  "person": "2MSG"},
    "netta":   {"pos": "PRON", "gloss": "él",          "person": "3MSG"},
    "nettat":  {"pos": "PRON", "gloss": "ella",        "person": "3FSG"},

    # Partículas
    "ddarwi":  {"pos": "ADV",  "gloss": "juntos"},
    "εawd":    {"pos": "ADV",  "gloss": "de nuevo"},
    "sin":     {"pos": "NUM",  "gloss": "dos"},
    "kraḍ":    {"pos": "NUM",  "gloss": "tres"},
}

# --- Patrones morfológicos por prefijos nominales ---
NOMINAL_PATTERNS = [
    (r"^a[^aeiou]",   "NOUN", "M.libre"),   # a + C... → masculino libre
    (r"^ta.+t$",      "NOUN", "F.libre"),   # ta...t   → femenino libre
    (r"^i[^aeiou]",   "NOUN", "M.plural"),  # i + C... → plural masculino
    (r"^ti.+in$",     "NOUN", "F.plural"),  # ti...in  → plural femenino
    (r"^u[^aeiou]",   "NOUN", "M.anexo"),   # u + C... → estado anexo masc.
    (r"^w[aeiou]",    "NOUN", "M.anexo"),   # w + V... → estado anexo masc.
]

# --- Patrones verbales por aspecto ---
VERBAL_PATTERNS = [
    (r"^tt.+",  "VERB", "IMPERF"),    # tt- → imperfectivo
    (r"^ss.+",  "VERB", "CAUS"),      # ss- → causativo
    (r"^mm.+",  "VERB", "RECIP"),     # mm- → recíproco
    (r"^m.+",   "VERB", "PASS"),      # m-  → pasivo/medio
    (r"^tt.+",  "VERB", "REFL"),      # tt- → reflexivo (contexto)
]


class TarifitMorphAnalyzer:
    """
    Analizador morfológico para Tarifit.
    Combina lexicón, patrones nominales y verbales.
    """

    def __init__(self, lexicon, nominal_patterns, verbal_patterns):
        self.lexicon   = lexicon
        self.nom_pats  = nominal_patterns
        self.verb_pats = verbal_patterns

    def analyze(self, token: str, clitic_label: str = None):
        """
        Analiza un token y devuelve su análisis morfológico completo.
        Si viene con etiqueta de clítico, la respeta.
        """
        result = {
            "token":  token,
            "pos":    "UNK",
            "gloss":  "?",
            "morph":  {},
            "source": "unknown"
        }

        # 1. Si tiene etiqueta de clítico, usarla directamente
        if clitic_label and clitic_label not in ("ROOT", "UNK"):
            result["pos"]    = self._clitic_to_pos(clitic_label)
            result["morph"]  = {"clitic_type": clitic_label}
            result["source"] = "clitic_dict"
            result["gloss"]  = self._clitic_gloss(token)
            return result

        # 2. Buscar en el lexicón
        if token in self.lexicon:
            entry = self.lexicon[token]
            result.update(entry)
            result["source"] = "lexicon"
            return result

        # 3. Aplicar patrones nominales
        for pattern, pos, morph in self.nom_pats:
            if re.match(pattern, token):
                result["pos"]    = pos
                result["morph"]  = {"nominal_state": morph}
                result["source"] = "pattern_nominal"
                result["gloss"]  = f"[{morph}]"
                return result

        # 4. Aplicar patrones verbales
        for pattern, pos, aspect in self.verb_pats:
            if re.match(pattern, token):
                result["pos"]    = pos
                result["morph"]  = {"aspect": aspect}
                result["source"] = "pattern_verbal"
                result["gloss"]  = f"[{aspect}]"
                return result

        # 5. Heurística final por longitud
        if len(token) >= 3:
            result["pos"]    = "VERB"
            result["source"] = "heuristic"
        else:
            result["pos"]    = "PART"
            result["source"] = "heuristic"

        return result

    def _clitic_to_pos(self, label: str):
        mapping = {
            "PREP": "ADP", "NEG": "PART", "NEG2": "PART",
            "GEN":  "ADP", "DAT": "ADP",  "STATE": "DET",
            "DIR":  "PART","DEM": "DET",  "ANAPH": "PART",
        }
        if label.startswith("PRO"): return "PRON"
        return mapping.get(label, "PART")

    def _clitic_gloss(self, token: str):
        glosses = {
            "ur":"NEG", "ara":"NEG2", "d":"and/COM",
            "s":"toward","n":"of","i":"to/DAT",
            "γur":"at/LOC","xf":"on/above","ger":"between",
            "iyi":"me","ak":"you(m)","as":"him/her",
            "aγ":"us","nni":"that(DEM)","nn":"ANAPH",
        }
        return glosses.get(token, f"[{token}]")

    def analyze_sentence(self, sentence: str, tokenizer):
        """Pipeline completo: tokenizar + analizar."""
        tokens_with_labels = tokenizer.tokenize_sentence(sentence)
        analyses = []
        for forma, label in tokens_with_labels:
            analysis = self.analyze(forma, label)
            analyses.append(analysis)
        return analyses


# --- Instanciar el analizador ---
analyzer = TarifitMorphAnalyzer(LEXICON, NOMINAL_PATTERNS, VERBAL_PATTERNS)

print("✅ Analizador morfológico listo")

✅ Analizador morfológico listo


In [9]:
# ============================================================
# Demostración del analizador morfológico completo
# ============================================================

test_sentences = [
    "ur ffiγ ara",
    "arba čča axxam",
    "γurnekkni taddart amuqqran",
]

glosas_es = [
    "No salí",
    "El niño comió en casa",
    "En nuestra aldea grande",
]

for sent, glosa in zip(test_sentences, glosas_es):
    print("=" * 58)
    print(f"📝 Oración: '{sent}'")
    print(f"🌍 Español: '{glosa}'")
    print("-" * 58)
    print(f"{'TOKEN':<14} {'POS':<8} {'GLOSA':<16} {'FUENTE'}")
    print("-" * 58)

    analyses = analyzer.analyze_sentence(sent, tarifit_tok)
    for a in analyses:
        print(f"{a['token']:<14} {a['pos']:<8} "
              f"{str(a['gloss']):<16} {a['source']}")
    print()

📝 Oración: 'ur ffiγ ara'
🌍 Español: 'No salí'
----------------------------------------------------------
TOKEN          POS      GLOSA            FUENTE
----------------------------------------------------------
ur             NOUN     [M.anexo]        pattern_nominal
ffiγ           VERB     salir            lexicon
ar             NOUN     [M.libre]        pattern_nominal
a              DET      [a]              clitic_dict

📝 Oración: 'arba čča axxam'
🌍 Español: 'El niño comió en casa'
----------------------------------------------------------
TOKEN          POS      GLOSA            FUENTE
----------------------------------------------------------
arb            NOUN     [M.libre]        pattern_nominal
a              DET      [a]              clitic_dict
čč             PART     ?                heuristic
a              DET      [a]              clitic_dict
axx            NOUN     [M.libre]        pattern_nominal
am             PRON     [am]             clitic_dict

📝 Oración: 'γurne

In [14]:
# ============================================================
# PASO 5 — Segmentador de morfemas para Tarifit
# Ejemplo morfologico:
# ur + i + ss + krs + as + d + ara
# NEG  1SG CAUS raiz 3OBJ VEN NEG2
# Significado: "no se lo desate hacia aca"
# ============================================================

import re

class MorphemeSegmenter:

    NOMINAL_PREFIXES = [
        ("tit", "F.PL.ANEXO"), ("ti", "F.PL.LIBRE"),
        ("ta",  "F.SG.LIBRE"), ("tu", "F.SG.ANEXO"),
        ("i",   "M.PL.LIBRE"), ("wa", "M.SG.ANEXO"),
        ("a",   "M.SG.LIBRE"), ("u",  "M.SG.ANEXO"),
    ]

    NOMINAL_SUFFIXES = [
        ("wen", "M.PL"), ("win", "M.PL"),
        ("in",  "PL"),   ("en",  "PL"),
        ("t",   "F.SG"),
    ]

    VERBAL_PREFIXES = [
        ("ttw", "PASS.IMPERF"), ("tt", "IMPERF"),
        ("ss",  "CAUS"),        ("sw", "CAUS.ALT"),
        ("mm",  "RECIP"),       ("m",  "PASS"),
        ("nn",  "INTENS"),
    ]

    PERSON_SUFFIXES = {
        "nt": "3FPL",
        "gh": "1SG",
        "n":  "1PL",
        "m":  "2MPL",
        "t":  "2SG",
    }

    def segment_noun(self, noun):
        result = {
            "original": noun,
            "prefix": None, "prefix_label": None,
            "root": noun,
            "suffix": None, "suffix_label": None,
            "consonantal_root": None,
        }
        working = noun.lower()
        for prefix, label in self.NOMINAL_PREFIXES:
            if working.startswith(prefix) and len(working) > len(prefix) + 1:
                result["prefix"] = prefix
                result["prefix_label"] = label
                working = working[len(prefix):]
                break
        for suffix, label in self.NOMINAL_SUFFIXES:
            if working.endswith(suffix) and len(working) > len(suffix) + 1:
                result["suffix"] = suffix
                result["suffix_label"] = label
                working = working[:-len(suffix)]
                break
        result["root"] = working
        result["consonantal_root"] = re.sub(r'[aeiou]', '', working) or working
        return result

    def segment_verb(self, verb):
        result = {
            "original": verb,
            "deriv_prefix": None, "deriv_label": None,
            "root": verb,
            "person_suffix": None, "person_label": None,
            "consonantal_root": None,
        }
        working = verb.lower()
        for prefix, label in self.VERBAL_PREFIXES:
            if working.startswith(prefix) and len(working) > len(prefix) + 1:
                result["deriv_prefix"] = prefix
                result["deriv_label"] = label
                working = working[len(prefix):]
                break
        for suffix, label in self.PERSON_SUFFIXES.items():
            if working.endswith(suffix) and len(working) > len(suffix):
                result["person_suffix"] = suffix
                result["person_label"] = label
                working = working[:-len(suffix)]
                break
        result["root"] = working
        result["consonantal_root"] = re.sub(r'[aeiou]', '', working) or working
        return result

    def display_segmentation(self, word, word_type="auto"):
        if word_type == "auto":
            word_type = "noun" if re.match(r'^(a|ta|i|ti|u|tu)', word) else "verb"
        print(f"\n--- Segmentacion: '{word}' [{word_type.upper()}]")
        if word_type == "noun":
            seg = self.segment_noun(word)
            parts = []
            if seg["prefix"]:
                parts.append(f"[{seg['prefix']}]={seg['prefix_label']}")
            parts.append(f"[{seg['root']}]=ROOT")
            if seg["suffix"]:
                parts.append(f"[{seg['suffix']}]={seg['suffix_label']}")
        else:
            seg = self.segment_verb(word)
            parts = []
            if seg["deriv_prefix"]:
                parts.append(f"[{seg['deriv_prefix']}]={seg['deriv_label']}")
            parts.append(f"[{seg['root']}]=ROOT")
            if seg["person_suffix"]:
                parts.append(f"[{seg['person_suffix']}]={seg['person_label']}")
        print("  " + "  +  ".join(parts))
        print(f"  Raiz consonantica: /{seg['consonantal_root']}/")
        return seg


# --- Instanciar y probar ---
segmenter = MorphemeSegmenter()

nombres = [("axxam","casa"), ("tafunast","vaca"), ("ixxamen","casas")]
verbos  = [("ffiγ","sali"), ("sskrs","hizo desatar"), ("kkreγ","me levante")]

print("===== NOMBRES =====")
for forma, glosa in nombres:
    segmenter.display_segmentation(forma, "noun")
    print(f"  Español: {glosa}")

print("\n===== VERBOS =====")
for forma, glosa in verbos:
    segmenter.display_segmentation(forma, "verb")
    print(f"  Español: {glosa}")

===== NOMBRES =====

--- Segmentacion: 'axxam' [NOUN]
  [a]=M.SG.LIBRE  +  [xxam]=ROOT
  Raiz consonantica: /xxm/
  Español: casa

--- Segmentacion: 'tafunast' [NOUN]
  [ta]=F.SG.LIBRE  +  [funas]=ROOT  +  [t]=F.SG
  Raiz consonantica: /fns/
  Español: vaca

--- Segmentacion: 'ixxamen' [NOUN]
  [i]=M.PL.LIBRE  +  [xxam]=ROOT  +  [en]=PL
  Raiz consonantica: /xxm/
  Español: casas

===== VERBOS =====

--- Segmentacion: 'ffiγ' [VERB]
  [ffiγ]=ROOT
  Raiz consonantica: /ffγ/
  Español: sali

--- Segmentacion: 'sskrs' [VERB]
  [ss]=CAUS  +  [krs]=ROOT
  Raiz consonantica: /krs/
  Español: hizo desatar

--- Segmentacion: 'kkreγ' [VERB]
  [kkreγ]=ROOT
  Raiz consonantica: /kkrγ/
  Español: me levante


In [15]:
# ============================================================
# Demostración de segmentación morfológica completa
# ============================================================

# --- Nombres para segmentar ---
nombres = [
    ("axxam",    "casa"),
    ("tafunast", "vaca"),
    ("ixxamen",  "casas"),
    ("tifunasin","vacas"),
    ("tamγart",  "mujer"),
    ("arba",     "niño"),
]

# --- Verbos para segmentar ---
verbos = [
    ("ffiγ",   "salí"),
    ("sskrs",  "hizo desatar (CAUS)"),
    ("ttili",  "estar.IMPERF"),
    ("mmγar",  "cortarse.RECIP"),
    ("kkreγ",  "me levanté"),
]

print("=" * 55)
print("📊 SEGMENTACIÓN DE NOMBRES TARIFIT")
print("=" * 55)
for forma, glosa in nombres:
    seg = segmenter.display_segmentation(forma, "noun")
    print(f"  Español: '{glosa}'")

print("\n" + "=" * 55)
print("📊 SEGMENTACIÓN DE VERBOS TARIFIT")
print("=" * 55)
for forma, glosa in verbos:
    seg = segmenter.display_segmentation(forma, "verb")
    print(f"  Español: '{glosa}'")

📊 SEGMENTACIÓN DE NOMBRES TARIFIT

--- Segmentacion: 'axxam' [NOUN]
  [a]=M.SG.LIBRE  +  [xxam]=ROOT
  Raiz consonantica: /xxm/
  Español: 'casa'

--- Segmentacion: 'tafunast' [NOUN]
  [ta]=F.SG.LIBRE  +  [funas]=ROOT  +  [t]=F.SG
  Raiz consonantica: /fns/
  Español: 'vaca'

--- Segmentacion: 'ixxamen' [NOUN]
  [i]=M.PL.LIBRE  +  [xxam]=ROOT  +  [en]=PL
  Raiz consonantica: /xxm/
  Español: 'casas'

--- Segmentacion: 'tifunasin' [NOUN]
  [ti]=F.PL.LIBRE  +  [funas]=ROOT  +  [in]=PL
  Raiz consonantica: /fns/
  Español: 'vacas'

--- Segmentacion: 'tamγart' [NOUN]
  [ta]=F.SG.LIBRE  +  [mγar]=ROOT  +  [t]=F.SG
  Raiz consonantica: /mγr/
  Español: 'mujer'

--- Segmentacion: 'arba' [NOUN]
  [a]=M.SG.LIBRE  +  [rba]=ROOT
  Raiz consonantica: /rb/
  Español: 'niño'

📊 SEGMENTACIÓN DE VERBOS TARIFIT

--- Segmentacion: 'ffiγ' [VERB]
  [ffiγ]=ROOT
  Raiz consonantica: /ffγ/
  Español: 'salí'

--- Segmentacion: 'sskrs' [VERB]
  [ss]=CAUS  +  [krs]=ROOT
  Raiz consonantica: /krs/
  Español: 'hi

In [16]:
# ============================================================
# Sistema de raíces consonánticas en Tarifit
# Muestra cómo una misma raíz genera múltiples palabras
# ============================================================

RAICES = {
    "KKR": {
        "desc": "levantarse / recoger",
        "formas": [
            ("kker",    "VERB.AOR",  "levantarse"),
            ("kkreγ",   "VERB.PERF.1SG", "me levanté"),
            ("takkart", "NOUN.F",    "acción de levantarse"),
            ("ssekker", "VERB.CAUS", "levantar a alguien"),
        ]
    },
    "FFΓ": {
        "desc": "salir",
        "formas": [
            ("ffiγ",    "VERB.PERF.1SG", "salí"),
            ("ffuγ",    "VERB.AOR",      "salir"),
            ("tifγa",   "NOUN.F",        "salida"),
            ("ssufuγ",  "VERB.CAUS",     "sacar"),
        ]
    },
    "XXM": {
        "desc": "casa / habitar",
        "formas": [
            ("axxam",   "NOUN.M.SG",  "casa"),
            ("ixxamen", "NOUN.M.PL",  "casas"),
            ("uxxam",   "NOUN.M.ANX", "casa (estado anexo)"),
        ]
    },
}

print("=" * 58)
print("🌳 FAMILIAS DE PALABRAS POR RAÍZ CONSONÁNTICA")
print("=" * 58)

for raiz, data in RAICES.items():
    print(f"\n📌 Raíz: /{raiz}/  →  {data['desc']}")
    print(f"  {'FORMA':<14} {'CATEGORÍA':<16} GLOSA")
    print(f"  {'─'*12} {'─'*14} {'─'*14}")
    for forma, cat, glosa in data["formas"]:
        print(f"  {forma:<14} {cat:<16} '{glosa}'")

🌳 FAMILIAS DE PALABRAS POR RAÍZ CONSONÁNTICA

📌 Raíz: /KKR/  →  levantarse / recoger
  FORMA          CATEGORÍA        GLOSA
  ──────────── ────────────── ──────────────
  kker           VERB.AOR         'levantarse'
  kkreγ          VERB.PERF.1SG    'me levanté'
  takkart        NOUN.F           'acción de levantarse'
  ssekker        VERB.CAUS        'levantar a alguien'

📌 Raíz: /FFΓ/  →  salir
  FORMA          CATEGORÍA        GLOSA
  ──────────── ────────────── ──────────────
  ffiγ           VERB.PERF.1SG    'salí'
  ffuγ           VERB.AOR         'salir'
  tifγa          NOUN.F           'salida'
  ssufuγ         VERB.CAUS        'sacar'

📌 Raíz: /XXM/  →  casa / habitar
  FORMA          CATEGORÍA        GLOSA
  ──────────── ────────────── ──────────────
  axxam          NOUN.M.SG        'casa'
  ixxamen        NOUN.M.PL        'casas'
  uxxam          NOUN.M.ANX       'casa (estado anexo)'


In [18]:
# ============================================================
# PASO 6 — Visualizacion con displaCy
#
# Como Tarifit no tiene modelo preentrenado, construimos
# el objeto Doc de spaCy manualmente con nuestro analisis.
# ============================================================

import spacy
from spacy.tokens import Doc
from spacy import displacy

# Crear modelo vacio base
nlp = spacy.blank("xx")

# ============================================================
# Funcion: convertir nuestro analisis al formato Doc de spaCy
# ============================================================

# Colores personalizados por categoria morfologica
COLORS = {
    "VERB":  "#85C1E9",   # azul claro
    "NOUN":  "#82E0AA",   # verde claro
    "PREP":  "#F8C471",   # naranja
    "NEG":   "#F1948A",   # rojo claro
    "NEG2":  "#F1948A",   # rojo claro
    "PRON":  "#C39BD3",   # morado claro
    "ADJ":   "#76D7C4",   # turquesa
    "ADV":   "#AED6F1",   # azul palido
    "ROOT":  "#D5DBDB",   # gris claro
    "PART":  "#FAD7A0",   # amarillo
    "ADP":   "#F0B27A",   # salmon
    "DET":   "#A9CCE3",   # azul grisaceo
    "NUM":   "#A3E4D7",   # menta
    "UNK":   "#EAECEE",   # gris
}

def build_spacy_doc(sentence, tokenizer, analyzer):
    """
    Construye un Doc de spaCy a partir de una oracion Tarifit.
    Usa nuestro tokenizer y analizador personalizados.
    """
    # Paso 1: tokenizar con nuestro sistema
    tokens_labels = tokenizer.tokenize_sentence(sentence)

    # Paso 2: analizar morfologicamente
    analyses = []
    for forma, label in tokens_labels:
        analysis = analyzer.analyze(forma, label)
        analyses.append(analysis)

    # Paso 3: extraer formas y etiquetas POS
    words = [a["token"] for a in analyses]
    pos_tags = [a["pos"] for a in analyses]

    # Paso 4: construir Doc manualmente
    doc = Doc(nlp.vocab, words=words)

    # Paso 5: asignar POS a cada token
    for token, pos in zip(doc, pos_tags):
        token.pos_ = pos

    return doc, analyses


def visualize_morphology(sentence, tokenizer, analyzer):
    """
    Visualiza el analisis morfologico de una oracion Tarifit
    usando displaCy en modo entidades (ent).
    """
    doc, analyses = build_spacy_doc(sentence, tokenizer, analyzer)

    # Construir entidades para displaCy
    ents = []
    char_offset = 0
    for analysis in analyses:
        token_text = analysis["token"]
        start = char_offset
        end   = char_offset + len(token_text)
        label = analysis["pos"]
        ents.append({
            "start": start,
            "end":   end,
            "label": label
        })
        char_offset = end + 1  # +1 por el espacio entre tokens

    # Datos para displaCy
    ex = [{
        "text":  " ".join(a["token"] for a in analyses),
        "ents":  ents,
        "title": None
    }]

    options = {
        "ents":   list(COLORS.keys()),
        "colors": COLORS
    }

    print(f"\nOracion: '{sentence}'")
    print("Analisis token a token:")
    print(f"  {'TOKEN':<14} {'POS':<8} {'GLOSA'}")
    print(f"  {'---':<14} {'---':<8} {'---'}")
    for a in analyses:
        print(f"  {a['token']:<14} {a['pos']:<8} {a.get('gloss','?')}")

    print("\nVisualizacion displaCy:")
    displacy.render(ex, style="ent", manual=True,
                    options=options, jupyter=True)


print("Funciones de visualizacion listas.")

Funciones de visualizacion listas.


In [19]:
# ============================================================
# Visualizar oraciones de ejemplo en Tarifit
# ============================================================

oraciones = [
    "ur ffiγ ara",
    "arba čča axxam",
    "sskrs as tafunast",
    "γur nekkni taddart amuqqran",
]

for oracion in oraciones:
    visualize_morphology(oracion, tarifit_tok, analyzer)
    print("=" * 55)


Oracion: 'ur ffiγ ara'
Analisis token a token:
  TOKEN          POS      GLOSA
  ---            ---      ---
  ur             NOUN     [M.anexo]
  ffiγ           VERB     salir
  ar             NOUN     [M.libre]
  a              DET      [a]

Visualizacion displaCy:



Oracion: 'arba čča axxam'
Analisis token a token:
  TOKEN          POS      GLOSA
  ---            ---      ---
  arb            NOUN     [M.libre]
  a              DET      [a]
  čč             PART     ?
  a              DET      [a]
  axx            NOUN     [M.libre]
  am             PRON     [am]

Visualizacion displaCy:



Oracion: 'sskrs as tafunast'
Analisis token a token:
  TOKEN          POS      GLOSA
  ---            ---      ---
  s              ADP      toward
  s              ADP      toward
  krs            VERB     ?
  as             NOUN     [M.libre]
  t              DET      [t]
  afunast        NOUN     [M.libre]

Visualizacion displaCy:



Oracion: 'γur nekkni taddart amuqqran'
Analisis token a token:
  TOKEN          POS      GLOSA
  ---            ---      ---
  γur            VERB     ?
  n              ADP      of
  ekkni          VERB     ?
  t              DET      [t]
  addart         NOUN     [M.libre]
  amuqqran       ADJ      grande

Visualizacion displaCy:


In [20]:
# ============================================================
# Modo DEP: arbol de dependencias simplificado
# Asignamos relaciones sintacticas manualmente
# para mostrar la estructura Sujeto-Verbo-Objeto de Tarifit
# ============================================================

def visualize_dependencies(sentence, tokenizer, analyzer):
    """
    Muestra un arbol de dependencias simplificado
    para una oracion Tarifit.
    """
    doc, analyses = build_spacy_doc(sentence, tokenizer, analyzer)

    words = [a["token"] for a in analyses]
    pos_list = [a["pos"] for a in analyses]

    # Encontrar indice del verbo principal (ROOT)
    root_idx = 0
    for i, pos in enumerate(pos_list):
        if pos == "VERB":
            root_idx = i
            break

    # Asignar dependencias automaticamente
    arcs = []
    word_data = []

    for i, (word, pos) in enumerate(zip(words, pos_list)):
        if i == root_idx:
            dep = "ROOT"
            head = i
        elif pos in ("NEG", "NEG2", "PART"):
            dep = "neg"
            head = root_idx
        elif pos in ("NOUN",):
            dep = "obj" if i > root_idx else "nsubj"
            head = root_idx
        elif pos in ("ADP", "PREP"):
            dep = "case"
            head = min(i + 1, len(words) - 1)
        elif pos == "ADJ":
            dep = "amod"
            head = max(i - 1, 0)
        elif pos == "PRON":
            dep = "obj"
            head = root_idx
        else:
            dep = "dep"
            head = root_idx

        word_data.append({
            "text": word,
            "tag":  pos,
            "dep":  dep,
            "head": head
        })

        if i != root_idx:
            if i < head:
                arcs.append({
                    "start": i,
                    "end":   head,
                    "label": dep,
                    "dir":   "left"
                })
            else:
                arcs.append({
                    "start": head,
                    "end":   i,
                    "label": dep,
                    "dir":   "right"
                })

    ex = [{
        "words": word_data,
        "arcs":  arcs
    }]

    options = {
        "compact":  True,
        "bg":       "#f5f5f5",
        "color":    "#2c3e50",
        "font":     "monospace"
    }

    print(f"\nArbol de dependencias: '{sentence}'")
    displacy.render(ex, style="dep", manual=True,
                    options=options, jupyter=True)


# --- Probar con oraciones ---
oraciones_dep = [
    "ur ffiγ ara",
    "arba čča axxam",
]

for oracion in oraciones_dep:
    visualize_dependencies(oracion, tarifit_tok, analyzer)
    print("=" * 55)


Arbol de dependencias: 'ur ffiγ ara'



Arbol de dependencias: 'arba čča axxam'


In [23]:
import spacy
from spacy.tokens import Doc
from spacy import displacy

nlp = spacy.blank("xx")

COLORS = {
    "VERB": "#85C1E9",
    "NOUN": "#82E0AA",
    "PREP": "#F8C471",
    "NEG":  "#F1948A",
    "NEG2": "#F1948A",
    "PRON": "#C39BD3",
    "ADJ":  "#76D7C4",
    "ADV":  "#AED6F1",
    "ROOT": "#D5DBDB",
    "PART": "#FAD7A0",
    "ADP":  "#F0B27A",
    "DET":  "#A9CCE3",
    "NUM":  "#A3E4D7",
    "UNK":  "#EAECEE",
}

def build_spacy_doc(sentence, tokenizer, analyzer):
    tokens_labels = tokenizer.tokenize_sentence(sentence)
    analyses = []
    for forma, label in tokens_labels:
        analysis = analyzer.analyze(forma, label)
        analyses.append(analysis)
    words = [a["token"] for a in analyses]
    doc = Doc(nlp.vocab, words=words)
    for token, a in zip(doc, analyses):
        token.pos_ = a["pos"]
    return doc, analyses

def visualize_morphology(sentence, tokenizer, analyzer):
    doc, analyses = build_spacy_doc(sentence, tokenizer, analyzer)
    ents = []
    char_offset = 0
    for analysis in analyses:
        token_text = analysis["token"]
        start = char_offset
        end   = char_offset + len(token_text)
        ents.append({"start": start, "end": end, "label": analysis["pos"]})
        char_offset = end + 1

    ex = [{
        "text": " ".join(a["token"] for a in analyses),
        "ents": ents,
        "title": None
    }]
    options = {"ents": list(COLORS.keys()), "colors": COLORS}

    print(f"\nOracion: '{sentence}'")
    print(f"  {'TOKEN':<14} {'POS':<8} {'GLOSA'}")
    print(f"  {'---':<14} {'---':<8} {'---'}")
    for a in analyses:
        print(f"  {a['token']:<14} {a['pos']:<8} {a.get('gloss','?')}")

    displacy.render(ex, style="ent", manual=True,
                    options=options, jupyter=True)

print("Funciones de visualizacion listas.")

Funciones de visualizacion listas.
